In [ ]:
from custom.util.data_manipulation import load_and_process_data, balance_dataset
from custom.models.ensemble import fit_ensemble_model, classify_then_regress
import json

_, _, numeric_features, categorical_features, df_train = load_and_process_data("data/claims_train.csv", True, "standard", True)
X_train, y_train, df_train = balance_dataset(df_train, numeric_features, categorical_features)




In [38]:
X_test, y_test, numeric_features, categorical_features, df_test = load_and_process_data("data/claims_test.csv", True, "standard", True)

In [24]:
with open("best_hyperparameters.json", "r") as f:
    hyperparameters_regression = json.load(f)
with open("best_hyperparameters_classification.json", "r") as f:
    hyperparameters_classification = json.load(f)

mlp_dict = {}
for model_name, params in hyperparameters_regression.items():
    if model_name == "MLP":
        for param_name, param_value in params.items():
            mlp_dict.setdefault(param_name.split("__")[1], param_value)
            mlp_dict[param_name.split("__")[1]] = param_value
        hyperparameters_regression[model_name] = mlp_dict

mlp_dict = {}
for model_name, params in hyperparameters_classification.items():
    if model_name == "MLP":
        for param_name, param_value in params.items():
            mlp_dict.setdefault(param_name.split("__")[1], param_value)
            mlp_dict[param_name.split("__")[1]] = param_value
        hyperparameters_classification[model_name] = mlp_dict

In [33]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPRegressor, MLPClassifier

classifiers = [
    ("rfc", RandomForestClassifier(**hyperparameters_classification["RandomForest"])),
    ("gbc", GradientBoostingClassifier(**hyperparameters_classification["GradientBoosting"])),
    ("mlpc", MLPClassifier(max_iter= 2000, **hyperparameters_classification["MLP"]))
]

regressors = [
    ("rfr", RandomForestRegressor(**hyperparameters_regression["RandomForest"])),
    ("gbr", GradientBoostingRegressor(**hyperparameters_regression["GradientBoosting"])),
    ("mlpr", MLPRegressor(max_iter= 2000,**hyperparameters_regression["MLP"]))
]

classify, regress = fit_ensemble_model(X_train, y_train, base_classification_models=classifiers, base_regrssion_models=regressors, verbose=3)

Fitting regression ensembles...
Fitting classification ensembles...


In [35]:
regress, classify = classify, regress

In [39]:
y_pred_test = classify_then_regress(X_test, classification_model=classify, regression_model=regress, threshold=0.5)

In [ ]:
from sklearn.metrics import mean_squared_error, classification_report

print("Test MSE:", mean_squared_error(y_test, y_pred_test))
y_test_class = (y_test > 0.5).astype(int)
y_pred_class = classify.predict(X_test)
print(classification_report(y_test_class, y_pred_class))

# Test MSE: 0.4096663389470959
#               precision    recall  f1-score   support

#            0       0.97      0.67      0.79    128473
#            1       0.08      0.56      0.14      6836

#     accuracy                           0.67    135309
#    macro avg       0.52      0.62      0.47    135309
# weighted avg       0.92      0.67      0.76    135309

Test MSE: 0.4096663389470959
              precision    recall  f1-score   support

           0       0.97      0.67      0.79    128473
           1       0.08      0.56      0.14      6836

    accuracy                           0.67    135309
   macro avg       0.52      0.62      0.47    135309
weighted avg       0.92      0.67      0.76    135309



In [41]:
import pickle

with open("final_classification_model.pkl", "wb") as f:
    pickle.dump(classify, f)

with open("final_regression_model.pkl", "wb") as f:
    pickle.dump(regress, f)